In [0]:
-- KPI: Revenue Growth % MoM
-- Purpose:
-- Calculates month-over-month revenue growth by comparing
-- the latest month with the previous month.

WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', order_date) AS sales_month,
        SUM(net_revenue) AS total_revenue
    FROM `end-to-end_pipeline`.gold.fact_sales
    GROUP BY DATE_TRUNC('month', order_date)
),

revenue_comparison AS (
    SELECT
        sales_month,
        total_revenue,
        LAG(total_revenue) OVER (
            ORDER BY sales_month
        ) AS previous_month_revenue
    FROM monthly_revenue
)

SELECT
    ROUND(
        ((total_revenue - previous_month_revenue)
        / NULLIF(previous_month_revenue, 0)) * 100,
        2
    ) AS revenue_growth_mom_pct
FROM revenue_comparison
WHERE previous_month_revenue IS NOT NULL
ORDER BY sales_month DESC
LIMIT 1;